In [ ]:
import logging
import pandas as pd
from pathlib import Path

from wildfire_susceptibility.config.loader import ConfigLoader
from wildfire_susceptibility.pipeline.dataset_builder import DatasetBuilder
from wildfire_susceptibility.pipeline.train import ModelTrainer
from wildfire_susceptibility.modeling.dataset_prep import DatasetPrep

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

CONFIG_PATH = "./wildfire_susceptibility/config/_working.yaml"  # or _working.yaml
cfg_obj = ConfigLoader.load(CONFIG_PATH)
cfg = cfg_obj.model_dump(mode="python")
cfg

In [ ]:
wf = DatasetBuilder(CONFIG_PATH)
dataset_paths = wf.run_full_pipeline()   # {season: {"train": path, "test": path}}

prep = DatasetPrep(cfg)
climate_vars = tuple(cfg["data_sources"]["haduk"]["sources"])
ref_path = Path(cfg["base"]["output_dir"]) / "topo_elevation.tif"

# for season, paths in dataset_paths.items():
#     raw = prep.load_train_test(paths)

#     df_train = prep.prepare_train(raw["train"], season, ref_path, climate_vars)
#     df_test  = prep.prepare_test(raw["test"], season, climate_vars)

In [3]:
all_results = {}

for season, paths in dataset_paths.items():
    raw = prep.load_train_test(paths)

    df_train = prep.prepare_train(raw["train"], season, ref_path, climate_vars)
    df_test  = prep.prepare_test(raw["test"], season, climate_vars)

    X_train = df_train.drop(columns=["label"])
    y_train = df_train["label"]

    X_test = df_test.drop(columns=["label"])
    y_test = df_test["label"]

    groups_train = None
    if cfg["modeling"].get("cv_strategy") in ("spatial", "both"):
        groups_train = prep.assign_spatial_blocks(X_train)

    feature_cols = [c for c in X_train.columns if not c.startswith("_")]
    trainer = ModelTrainer(cfg)

    season_results = {}
    for model_name in cfg["modeling"]["models"]:
        print(f"\n--- [{season}] training {model_name} ---")

        def _on_trial(m_name, trial_number, trial_value):
            print(f"  [{season}][{m_name}] trial {trial_number+1}/{cfg['modeling']['optuna_n_trials']} "
                  f"AUC={trial_value:.4f}")

        result = trainer.train_one(
            season, model_name,
            X_train[feature_cols], y_train,
            X_test[feature_cols], y_test,
            groups_train=groups_train,
            progress_callback=_on_trial,
        )

        print(f"  >>> standard CV AUC={result['cv_auc_standard']:.4f}", end="")
        if result["cv_auc_spatial"] is not None:
            gap = result['cv_auc_standard'] - result['cv_auc_spatial']
            print(f" | spatial CV AUC={result['cv_auc_spatial']:.4f} | optimism gap={gap:.4f}", end="")
        print(f" | val AUC={result['val_auc']:.4f} | val F1={result['val_f1']:.4f}")

        season_results[model_name] = result

    all_results[season] = season_results

print("\nAll training complete.")

In [ ]:
rows = []
for season, models in all_results.items():
    for model_name, r in models.items():
        rows.append({
            "season": season,
            "model": model_name,
            "cv_auc_standard": r["cv_auc_standard"],
            "cv_auc_spatial": r["cv_auc_spatial"],
            "val_auc": r["val_auc"],
            "val_f1": r["val_f1"],
        })
pd.DataFrame(rows).sort_values(["season", "cv_auc_standard"], ascending=[True, False])

In [ ]:
from wildfire_susceptibility.reporting import generate_all

generate_all(config_path=CONFIG_PATH, seasons=list(dataset_paths.keys()), categories=["models", "eda"])